# Recipe Recommender

Two approaches: **content-based** (similar descriptions via TF-IDF) and **collaborative** (users who liked this also liked that).

### Setup

In [1]:
import sys
import os
import logging
from pathlib import Path
import joblib

# Ensure project root is on the path
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO)

from src.models.recommender import (
    load_recipes, build_tfidf, recommend_similar,
    load_reviews, build_user_item_matrix, recommend_collaborative,
)

### Loading Recipe Data

In [2]:
recipes = load_recipes(limit=50000)
print(f"Loaded {len(recipes):,} recipes for recommendation")
print(f"Sample description: {recipes['Description'].iloc[0][:200]}")

INFO:src.models.recommender:Loaded 50,000 recipes for recommendation


Loaded 50,000 recipes for recommendation
Sample description: I searched and finally found this recipe on the internet. It is a copycat of the Bourbon Chicken sold in Chinese carry-outs in my hometown.  This recipe is so good that my sons gobble it up leaving me


### Content-Based — TF-IDF on Descriptions

In [3]:
tfidf, tfidf_matrix = build_tfidf(recipes)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Top features: {tfidf.get_feature_names_out()[:20]}")

INFO:src.models.recommender:TF-IDF matrix: (50000, 5000)


TF-IDF matrix shape: (50000, 5000)
Top features: ['05' '07' '08' '09' '10' '10 minutes' '10 years' '100' '11' '12' '13'
 '14' '15' '15 minutes' '15 years' '16' '18' '1998' '1999' '1st']


### Build the Similarity Function

In [4]:
print("=" * 60)
recommend_similar("Banana Bread", recipes, tfidf_matrix, n=10)

INFO:src.models.recommender:Finding recipes similar to: Best Banana Bread (Quick Breads)


,Name,RecipeCategory,AggregatedRating,ReviewCount,Similarity
1,Best Banana Bread,Quick Breads,5.0,2273.0,1.000000
42268,Best Banana Bread,Breads,5.0,7.0,1.000000
37152,Best Banana Bread Or Muffins,Quick Breads,4.5,7.0,0.846035
27782,Peanutty Chocolate Banana Bread,Breads,5.0,9.0,0.686566
40440,No-Butter Banana Bread,Quick Breads,5.0,7.0,0.686547
43472,Mom's Banana Bread,Breads,5.0,7.0,0.676841
35918,Famous Banana Bread,Quick Breads,5.0,8.0,0.648027
32269,Banana Bread,Quick Breads,5.0,8.0,0.636262
6415,Banana Bread,Quick Breads,5.0,28.0,0.636262
3535,Applesauce Banana Bread,Quick Breads,4.5,44.0,0.636262


### Collaborative Filtering

In [5]:
# Use top 20,000 most-reviewed recipes for collaborative filtering.
# The full 112K creates a similarity matrix too large for memory (~100GB).
# 20K keeps it at ~3.2GB — comfortable on 24GB RAM.
collab_recipes = recipes.nlargest(20000, "ReviewCount")
collab_recipe_ids = collab_recipes["RecipeId"].tolist()

reviews = load_reviews(min_ratings=5, recipe_ids=collab_recipe_ids)
print(f"\nUsing top {len(collab_recipes):,} recipes for collaborative filtering")

INFO:src.models.recommender:Loaded 710,845 ratings from 183,560 users (19,999 recipes)
INFO:src.models.recommender:Active users (5+ ratings): 18,756 — 501,139 ratings



Using top 20,000 recipes for collaborative filtering


### User-Item Matrix + Similarity

In [6]:
user_item, item_similarity = build_user_item_matrix(reviews)
print(f"User-Item matrix: {user_item.shape}")
print(f"Sparsity: {(user_item == 0).sum().sum() / user_item.size:.2%}")
print(f"Item similarity matrix: {item_similarity.shape}")

INFO:src.models.recommender:User-Item matrix: (18756, 19950)
INFO:src.models.recommender:Sparsity: 99.87%
INFO:src.models.recommender:Item similarity matrix: (19950, 19950)


User-Item matrix: (18756, 19950)
Sparsity: 99.87%
Item similarity matrix: (19950, 19950)


### Collaborative Recommender

In [7]:
popular_recipe = collab_recipes.iloc[0]
print(f"Collaborative recommendations for: {popular_recipe['Name']}")
recommend_collaborative(
    popular_recipe["RecipeId"], user_item, item_similarity, recipes, n=10
)

Collaborative recommendations for: Bourbon Chicken


,RecipeId,Name,RecipeCategory,AggregatedRating,ReviewCount,Similarity
19,28148,Oven-Fried Chicken Chimichangas,One Dish Meal,5.0,835.0,0.202644
4,39087,Creamy Cajun Chicken Pasta,Chicken Breast,5.0,1586.0,0.202625
2,27208,To Die for Crock Pot Roast,One Dish Meal,5.0,1692.0,0.193812
3,89204,Crock-Pot Chicken With Black Beans & Cream Cheese,One Dish Meal,4.5,1657.0,0.182563
35,95222,Pork Chops Yum-Yum,Pork,4.5,681.0,0.181687
8,22782,Jo Mama's World Famous Spaghetti,Spaghetti,5.0,1326.0,0.171082
9,32204,"""Whatever Floats Your Boat"" Brownies!",Bar Cookie,5.0,1284.0,0.167061
14,68955,Japanese Mum's Chicken,Chicken Thigh & Leg,5.0,973.0,0.163437
12,69173,Kittencal's Italian Melt-In-Your-Mouth Meatballs,Meat,5.0,1068.0,0.161989
15,33919,Creamy Burrito Casserole,Meat,5.0,940.0,0.161976


### Save the Recommender

In [8]:
joblib.dump({
    "tfidf": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "recipe_ids": recipes["RecipeId"].values,
    "recipe_names": recipes["Name"].values,
}, MODELS_DIR / "recommender_tfidf.joblib")

print("Saved content-based recommender")

Saved content-based recommender
